In [1]:
path = '../Thesis_Data/'

DATA_FILE = path + 'Final_Data/Final_Merged_Data.csv'
OUT_DIR   = path + 'HTS_Results/'

TARGET    = 'milk_kg'
FREQ      = 'MS'        # Month-Start — test-day records are ~monthly

In [2]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from hierarchicalforecast.utils import aggregate

## 2. Load Preprocessed Data


In [7]:
df = pd.read_csv(
    DATA_FILE,
    usecols=['Farm_Code', 'Animal_ID', 'dtt', TARGET],
    parse_dates=['dtt'],
)

df['Farm_Code']  = df['Farm_Code'].astype(str)
df['Animal_ID']  = df['Animal_ID'].astype(str)


print(f'Records  : {len(df):,}')
print(f'Animals  : {df["Animal_ID"].nunique():,}')
print(f'Farms    : {df["Farm_Code"].nunique():,}')
print(f'Date range: {df["dtt"].min().date()} → {df["dtt"].max().date()}')
df.head(5)

Records  : 671,655
Animals  : 81,119
Farms    : 314
Date range: 2013-01-02 → 2023-12-29


,Farm_Code,Animal_ID,dtt,milk_kg
0,521513,IT003990057639,2013-06-04,6.8
1,521513,IT003990057639,2013-07-02,8.0
2,521513,IT003990057639,2013-09-03,7.8
3,521513,IT003990057639,2013-10-02,8.2
4,521513,IT003990057639,2013-11-04,5.2


## 3. Resample to Monthly Frequency

Test-day records are irregular (~one per month per animal). We snap each record to the first day of its month (`MS`) and take the **mean** when multiple tests fall in the same month (rare after preprocessing).


In [8]:
# Snap test-day date to month start
df['ds'] = df['dtt'].dt.to_period('M').dt.to_timestamp()

# Aggregate: mean milk per animal per month
df_monthly = (
    df.groupby(['Farm_Code', 'Animal_ID', 'ds'])[TARGET]
    .mean()
    .reset_index()
    .rename(columns={TARGET: 'y'})
)

print(f'Monthly records (animal level): {len(df_monthly):,}')
print(f'Date range: {df_monthly["ds"].min().date()} → {df_monthly["ds"].max().date()}')
df_monthly.head()

Monthly records (animal level): 660,241
Date range: 2013-01-01 → 2023-12-01


,Farm_Code,Animal_ID,ds,y
0,1114232,IT019990445374,2013-01-01,8.0
1,1114232,IT019990445374,2013-02-01,7.4
2,1114232,IT019990445374,2013-03-01,6.0
3,1114232,IT019990445374,2013-04-01,7.4
4,1114232,IT019990445420,2013-01-01,11.6


## 4. Build the Hierarchical Panel (Y_df), Summing Matrix (S_df), and tags

In [12]:
df_hier = df_monthly.copy()
df_hier['Total'] = 'Total'
df_hier.head()
spec = [
    ['Total'],
    ['Total', 'Farm_Code'],
    ['Total', 'Farm_Code', 'Animal_ID'],
]
# aggregate() returns Y_df (long panel), S_df (summing matrix), tags (level → ids)
Y_df, S_df, tags = aggregate(df_hier, spec)

: 

In [ ]:
n_series = Y_df['unique_id'].nunique()
print(f'Total series : {n_series:,}')
for level, ids in tags.items():
    print(f'  {level:<12}: {len(ids):,} series')
print(f'S_df shape   : {S_df.shape}')
print(f'Y_df rows    : {len(Y_df):,}')
print(f'Date range   : {Y_df["ds"].min().date()} → {Y_df["ds"].max().date()}')

In [ ]:
all_ids    = Y_df['unique_id'].unique().tolist()
bottom_ids = tags[list(tags.keys())[-1]]   
farm_ids   = tags[list(tags.keys())[-2]]   

In [ ]:
os.makedirs(path+'/HTS_Res', exist_ok=True)

## Y_df
Y_df.to_parquet(path+'/HTS_Res/Y_df.parquet')
## S_df
S_df.to_parquet(path+'/HTS_Res/S_df.parquet')

## Tags
clean_tags = {level: list(ids) for level, ids in tags.items()}
with open(path+'/HTS_Res/tags.json', 'w') as f:
    json.dump(clean_tags, f, indent=4) 


## 5. Visualise the Hierarchy

Plot the Total series and a sample of farm-level series to inspect seasonality and trend.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Top: Total
tot = Y_df[Y_df['unique_id'] == 'Total'].sort_values('ds')
axes[0].plot(tot['ds'], tot['y'], color='#1a1a2e', linewidth=2)
axes[0].set_title('Total Herd — Monthly Milk Yield', fontsize=13, fontweight='bold')
axes[0].set_ylabel('kg')

# Middle: random sample of 5 farms
np.random.seed(3)
sample_farms_plot = np.random.choice(farm_ids, min(5, len(farm_ids)), replace=False)
for fid in sample_farms_plot:
    ts = Y_df[Y_df['unique_id'] == fid].sort_values('ds')
    axes[1].plot(ts['ds'], ts['y'], label=fid, linewidth=1)
axes[1].set_title('Sample of Farm-Level Series (5 farms)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('kg')
axes[1].legend(fontsize=7, ncol=3)

# Bottom: random sample of 5 animals from first sample farm
farm_code_sample = sample_farms_plot[1]
animal_ids_in_farm = [b for b in bottom_ids if b.startswith(farm_code_sample + '/')]
sample_animals_plot = np.random.choice(animal_ids_in_farm, min(10, len(animal_ids_in_farm)), replace=False)
for aid in sample_animals_plot:
    ts = Y_df[Y_df['unique_id'] == aid].sort_values('ds')
    axes[2].plot(ts['ds'], ts['y'], linewidth=1, alpha=0.8)
axes[2].set_title(f'Sample Animals from Farm {farm_code_sample}', fontsize=13, fontweight='bold')
axes[2].set_ylabel('kg')
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=30)

plt.tight_layout()
# plt.savefig(OUT_DIR + 'hierarchy_overview.png', dpi=150, bbox_inches='tight')
plt.show()
# print('Plot saved ✅')